# SGT runner — Colab Pro (L4 / A100)

Mounts Drive, clones the prism repo, and dispatches one or more pre-registered runs.
All scientific logic lives in `experiments/sgt/sgt_runner.py` — keep this notebook thin.

**Before running:** confirm `preregistration.md` is committed in the repo head.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf prism
!git clone --depth 1 https://github.com/humanaiconvention/prism
%cd /content/prism
!pip install -q -e . transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 bitsandbytes==0.44.1
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu124
# Colab's base image preinstalls torchvision/torchaudio at cu128; after we
# downgrade torch to cu124 the C++ ABI breaks (operator torchvision::nms).
# SGT does not import torchvision/torchaudio, so the simplest fix is to remove them.
!pip uninstall -q -y torchvision torchaudio


In [ ]:
import os, sys
sys.path.insert(0, '/content/prism/src')
RUNS_DIR = '/content/drive/MyDrive/sgt_runs/0p5b'
os.makedirs(RUNS_DIR, exist_ok=True)
MODEL = 'Qwen/Qwen2.5-0.5B'
TEACHER = 'Qwen/Qwen2.5-1.5B'   # for R4 only

## Dispatch — pick what to run this session

Each cell below is one config. Comment out completed runs. Locked seeds: 11, 23, 42.

In [ ]:
# Smoke test (10 min) — confirm pipeline before committing GPU-hours
!cd /content/prism/experiments/sgt && python sgt_runner.py \
  --regime R1 --seed 11 --model {MODEL} --generations 2 --samples_per_gen 200 \
  --out {RUNS_DIR}_smoke

In [ ]:
# Main 0.5B sweep — 5 regimes × 3 seeds. Run in batches to fit Colab quotas.
import subprocess
REGIMES = ['R1', 'R1_accum', 'R2', 'R3', 'R4']
SEEDS   = [11, 23, 42]
for regime in REGIMES:
    for seed in SEEDS:
        cmd = ['python', 'sgt_runner.py', '--regime', regime, '--seed', str(seed),
               '--model', MODEL, '--out', RUNS_DIR]
        if regime == 'R4': cmd += ['--teacher', TEACHER]
        print('===', ' '.join(cmd))
        subprocess.run(cmd, cwd='/content/prism/experiments/sgt', check=True)

In [ ]:
# Correction-fraction sweep at 0.5B (Rn_<frac>) — separate from the main sweep
for frac in [0.0, 0.25, 0.5, 0.75, 1.0]:
    for seed in [11, 23, 42]:
        cmd = ['python', 'sgt_runner.py', '--regime', f'Rn_{int(frac*100)}',
               '--seed', str(seed), '--model', MODEL, '--teacher', TEACHER,
               '--correction_frac', str(frac), '--out', f'{RUNS_DIR}/sweep']
        print('===', ' '.join(cmd))
        subprocess.run(cmd, cwd='/content/prism/experiments/sgt', check=True)